# Rolling, expanding i funkcje kumulatywne w pandas i polars

**Problem:** okna czasowe (`rolling`) i kumulatywne agregacje (`cumsum`, `expanding`) wyglądają na proste, dopóki dane są regularne. Przy prawdziwych danych z lukami (`window=N` licząc wiersze, nie dni) albo pojedynczym brakiem (`NaN` zatruwający całe okno) łatwo dostać liczbowo poprawny, ale merytorycznie błędny wynik — bez żadnego ostrzeżenia.

**Porównanie:**
- `.rolling(window=N)` — **N ostatnich wierszy**, niezależnie od tego, ile czasu kalendarzowego obejmują.
- `.rolling(window="ND")` — **N dni kalendarzowych**, niezależnie od tego, ile wierszy w nie wpada (wymaga `DatetimeIndex`).
- `.expanding()` — okno rosnące od początku serii do bieżącego wiersza ("wszystko dotychczas").
- `.cumsum()`/`.cummax()`/`.cumcount()` — pojedyncza wartość skumulowana, bez pojęcia "okna" o stałym rozmiarze.

**Kiedy stosować:** `window=N` gdy dane są regularne (jeden wiersz na dzień, bez luk) i faktycznie chodzi o "N ostatnich obserwacji"; `window="ND"` zawsze wtedy, gdy w grę wchodzi konkretny okres kalendarzowy, a dane mogą mieć luki.

## Setup

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

rng = np.random.default_rng(11)
dates = pd.date_range("2026-01-01", periods=10, freq="D")
sales = rng.normal(1000, 100, 10).round(1)

df = pd.DataFrame({"date": dates, "sales": sales}).set_index("date")
df_pl = pl.DataFrame({"date": dates, "sales": sales})

df

## Sekcja 1 — `.rolling()`: podstawy (okno = liczba wierszy)

Pierwsze `window - 1` wartości to `NaN` — nie ma jeszcze wystarczająco dużo poprzednich wierszy, żeby wypełnić okno.

In [ ]:
df["rolling_mean_3"] = df["sales"].rolling(window=3).mean()
df["rolling_sum_3"] = df["sales"].rolling(window=3).sum()
df[["sales", "rolling_mean_3", "rolling_sum_3"]]

## Sekcja 2 — `.expanding()`: okno rosnące od początku

W przeciwieństwie do `rolling`, `expanding` nigdy nie "zapomina" najstarszych wartości — każdy kolejny wynik uwzględnia wszystkie wiersze od początku serii do bieżącego. Przydatne np. do średniej narastającej "od początku roku do dziś".

In [ ]:
df["expanding_mean"] = df["sales"].expanding().mean()
df[["sales", "expanding_mean"]]

## Sekcja 3 — Funkcje kumulatywne: `cumsum`, `cummax`, `cumcount`

Nie mają pojęcia "okna" — każdy wynik to funkcja WSZYSTKICH poprzednich wierszy. `cumcount()` (dostępne po `groupby`) numeruje wiersze w obrębie grupy — przydatne np. do oznaczenia "pierwsza/druga/trzecia transakcja tego klienta".

In [ ]:
df["cumsum"] = df["sales"].cumsum()
df["cummax"] = df["sales"].cummax()
df[["sales", "cumsum", "cummax"]]

In [ ]:
orders = pd.DataFrame({
    "region": ["North", "North", "North", "South", "South"],
    "sales": [100, 200, 150, 300, 250],
})
orders["order_in_region"] = orders.groupby("region").cumcount() + 1  # +1, bo cumcount zaczyna od 0
orders

## Sekcja 4 — Okno oparte na czasie: `.rolling("ND")`

Zamiast liczby wierszy, `window="7D"` liczy **dni kalendarzowe** wstecz od bieżącego wiersza. Wymaga `DatetimeIndex`. To rozróżnienie ma kolosalne znaczenie przy danych z lukami — patrz Pułapka 1.

In [ ]:
df["rolling_3d"] = df["sales"].rolling(window="3D").mean()
df[["sales", "rolling_mean_3", "rolling_3d"]]

## Sekcja 5 — `min_periods` i `closed`

- **`min_periods`** — minimalna liczba nie-`NaN` obserwacji w oknie, żeby w ogóle policzyć wynik. `min_periods=1` daje wynik od pierwszego wiersza, zamiast czekać na pełne okno.
- **`closed`** — czy bieżący wiersz wlicza się do własnego okna. Domyślnie `"right"` (wlicza się) — dla niektórych zastosowań (np. "średnia z poprzednich dni, bez dzisiaj") potrzebne jest `"left"`.

In [ ]:
s = pd.Series([1, 2, 3, 4, 5], index=pd.date_range("2026-01-01", periods=5, freq="D"))

print("closed='right' (domyślne) - bieżący wiersz WLICZONY:")
print(s.rolling("3D", closed="right").sum())

print("\nclosed='left' - bieżący wiersz NIE wliczony (tylko poprzednie):")
print(s.rolling("3D", closed="left").sum())

## Sekcja 6 — Rolling per grupa

`.rolling()` na wyniku `.groupby()` liczy okno w obrębie każdej grupy osobno — przez `.transform()` z lambdą, żeby wynik miał kształt oryginalnego DataFrame.

In [ ]:
df_regions = pd.DataFrame({
    "region": ["North"] * 5 + ["South"] * 5,
    "date": list(pd.date_range("2026-01-01", periods=5, freq="D")) * 2,
    "sales": rng.normal(1000, 100, 10).round(1),
}).set_index("date")

df_regions["rolling_per_region"] = df_regions.groupby("region")["sales"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df_regions

## Sekcja 7 — polars: `rolling_mean`, `rolling_mean_by`, `cum_sum`, `.over()`

polars rozdziela to na dwie osobne metody: `rolling_mean()` liczy okno po liczbie wierszy, `rolling_mean_by()` po czasie kalendarzowym (odpowiednik `window="ND"` w pandas) — nazwa jasno mówi, z którym wariantem masz do czynienia, więc trudniej pomylić je przez przypadek.

In [ ]:
df_pl.with_columns([
    pl.col("sales").rolling_mean(window_size=3).alias("rolling_3_rows"),
    pl.col("sales").rolling_mean_by("date", window_size="3d").alias("rolling_3d"),
    pl.col("sales").cum_sum().alias("cumsum"),
    pl.col("sales").cum_max().alias("cummax"),
])

`.over("kolumna")` w polars to odpowiednik `groupby().transform()` z Sekcji 6 — działa też z funkcjami kumulatywnymi, nie tylko `sum`/`mean` zwracającymi jedną wartość na grupę.

In [ ]:
df_regions_pl = pl.DataFrame({
    "region": ["North"] * 5 + ["South"] * 5,
    "date": list(dates[:5]) * 2,
    "sales": rng.normal(1000, 100, 10).round(1),
})

df_regions_pl.with_columns(
    pl.col("sales").cum_sum().over("region").alias("cumsum_per_region")
)

## Sekcja 8 — Pułapki

### Pułapka 1 — `window=N` liczy WIERSZE, nie dni. Przy lukach w danych to nie to samo.

Dane poniżej mają luki (brak wpisów za kilka dni — typowa sytuacja: weekend bez transakcji). `rolling(window=7)` mimo to bierze "7 ostatnich wierszy", które w kalendarzu mogą rozciągać się na znacznie więcej niż 7 dni. `rolling(window="7D")` poprawnie trzyma się rzeczywistych 7 dni kalendarzowych — wynik na `2026-01-14` różni się zauważalnie między obiema metodami.

In [ ]:
dates_full = pd.date_range("2026-01-01", periods=20, freq="D")
keep_mask = np.ones(20, dtype=bool)
keep_mask[[4, 5, 11, 12]] = False  # symulacja brakujących dni (np. weekend bez sprzedaży)
dates_sparse = dates_full[keep_mask]
sales_sparse = rng.normal(1000, 100, len(dates_sparse)).round(1)

df_sparse = pd.DataFrame({"date": dates_sparse, "sales": sales_sparse}).set_index("date")
print(f"Wierszy: {len(df_sparse)}, na {len(dates_full)} dni kalendarzowych\n")

df_sparse["rolling_7_rows"] = df_sparse["sales"].rolling(window=7).mean()
df_sparse["rolling_7_days"] = df_sparse["sales"].rolling(window="7D").mean()
df_sparse[["sales", "rolling_7_rows", "rolling_7_days"]]

### Pułapka 2 — jeden brakujący wiersz (`NaN`) zatruwa CAŁE okno, w którym się znajdzie

Pojedyncza brakująca wartość w środku serii nie zostaje po prostu pominięta — każde okno, które ją obejmuje, zwraca `NaN` dla całej agregacji. Poniżej: 1 brakująca wartość źródłowa powoduje aż 5 wartości `NaN` w wyniku (2 z rozgrzewki na starcie okna + 3 zatrute przez brak).

In [ ]:
sales_with_gap = pd.Series(
    [100, 200, 150, np.nan, 250, 300, 280, 310, 295, 320.0],
    index=pd.date_range("2026-01-01", periods=10, freq="D"),
)

result = sales_with_gap.rolling(window=3).mean()
print(result)
print(f"\nLiczba NaN w wyniku: {result.isna().sum()} (a tylko 1 wartość źródłowa była NaN)")

## Podsumowanie

| Zadanie | pandas | polars |
|---|---|---|
| Okno po liczbie wierszy | `.rolling(window=N)` | `.rolling_mean(window_size=N)` |
| Okno po dniach kalendarzowych | `.rolling(window="ND")` (wymaga `DatetimeIndex`) | `.rolling_mean_by("kolumna_daty", window_size="Nd")` |
| Okno rosnące od początku serii | `.expanding()` | brak bezpośredniego odpowiednika — `.cum_sum()` itp. pełnią tę rolę |
| Suma/maksimum narastające | `.cumsum()` / `.cummax()` | `.cum_sum()` / `.cum_max()` |
| Numer wiersza w obrębie grupy | `.groupby("col").cumcount()` | `pl.int_range(pl.len()).over("col")` |
| Wynik od pierwszego wiersza (nie czekać na pełne okno) | `min_periods=1` | domyślne zachowanie przy `rolling_mean_by`; `min_samples` w `rolling_mean` |
| Czy bieżący wiersz wlicza się do własnego okna | `closed="right"/"left"/"both"/"neither"` | analogiczny parametr `closed` w funkcjach `rolling_*_by` |
| Okno/kumulacja w obrębie grupy | `.groupby("col")["x"].transform(lambda s: s.rolling(...))` | `pl.col("x").rolling_mean(...).over("col")` |

**Wniosek:** ten sam motyw co w notatkach o `merge`/`concat` i `groupby` — żadna z tych pułapek nie rzuca błędu. Wynik zawsze wygląda na policzony poprawnie; różnica ujawnia się dopiero przy porównaniu z tym, co *powinno* wyjść. Nawyk: przy danych z możliwymi lukami czasowymi zawsze pytaj, czy `window=N` naprawdę oznacza to, co zakładasz.